# Mini RAG Chatbot

## Project Overview

This project implements a simple **Retrieval-Augmented Generation (RAG)** chatbot
that answers questions using information retrieved from a PDF document.

### Knowledge Source

**A Beginner's Guide to Data & Analytics**



In [2]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.14.4 (main, Jun 18 2026, 14:25:02) [GCC 15.2.0]
Python executable: /home/nineleaps/mini-rag-chatbot/rag_env/bin/python


## 1. Import Required Libraries

The project uses:

- `pypdf` for PDF extraction
- `langchain` for document handling and text splitting
- `sentence-transformers` for embeddings
- `faiss-cpu` for vector search
- `langchain-openai` for LLM integration
- `python-dotenv` for securely loading the API key from `.env`

In [3]:
import os
import numpy as np

from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

print("All required libraries imported successfully.")

/home/nineleaps/mini-rag-chatbot/rag_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_29619/1481433549.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


All required libraries imported successfully.


## 2. Project Setup

The PDF is stored inside the `sample_data` folder.

Expected project structure:

```text
mini-rag-chatbot/
│
├── Mini_RAG_Chatbot.ipynb
├── README.md
├── requirements.txt
├── .env
├── .gitignore
├── sample_data/
│   └── data.pdf
└── screenshots/
```

In [4]:
pdf_path = "sample_data/data.pdf"

print("PDF path:", pdf_path)
print("PDF exists:", os.path.exists(pdf_path))

PDF path: sample_data/data.pdf
PDF exists: True


## 3. PDF Text Extraction

The first stage of the RAG pipeline is to read the PDF and extract
its text page by page.

Keeping the page number as metadata allows us to identify the source
page of retrieved information later.

In [5]:
reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

pages_text = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages_text.append({
        "page": page_number,
        "text": text
    })

print("Pages extracted:", len(pages_text))

Number of pages: 22
Pages extracted: 22


In [6]:
empty_pages = [
    page["page"]
    for page in pages_text
    if not page["text"].strip()
]

print("Pages with no extracted text:", empty_pages)

Pages with no extracted text: []


In [7]:
print(pages_text[0]["text"][:1000])

A Beginner’s Guide 
to Data & Analytics


In [8]:
documents = []

for page in pages_text:
    if page["text"].strip():
        documents.append(
            Document(
                page_content=page["text"],
                metadata={
                    "source": pdf_path,
                    "page": page["page"]
                }
            )
        )

print("Number of LangChain documents:", len(documents))
print("First document metadata:", documents[0].metadata)

Number of LangChain documents: 22
First document metadata: {'source': 'sample_data/data.pdf', 'page': 1}


## 4. Text Chunking

Large documents are split into smaller pieces called **chunks**.

Chunking helps the embedding model represent focused pieces of information
and improves the quality of semantic retrieval.

We use:

- `chunk_size = 800`
- `chunk_overlap = 100`

The overlap helps preserve context when information crosses a chunk boundary.

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 51


In [10]:
for i, chunk in enumerate(chunks[:5], start=1):
    print(f"\n--- Chunk {i} ---")
    print("Page:", chunk.metadata["page"])
    print("Characters:", len(chunk.page_content))
    print(chunk.page_content[:500])


--- Chunk 1 ---
Page: 1
Characters: 39
A Beginner’s Guide 
to Data & Analytics

--- Chunk 2 ---
Page: 2
Characters: 709
Contents
3 Data Science 
vs. Data Analytics: What’s 
the Difference?
7 Data Literacy 101: 
Familiarizing Yourself with 
the Data Landscape
11 Building Your Data & 
Analytical Skill Set
19 Which Data & Analytics 
Course Is Right for You?
Data is ubiquitous. It’s collected at every purchase made, flight taken, ad clicked, and 
social media post liked—which means it’s never been more accessible to organizations.
Yet, access to data isn’t all it takes to set a business on the path to success; it als

--- Chunk 3 ---
Page: 2
Characters: 758
data to drive decision-making. 
“In this world of big data, basic data literacy—the ability to analyze, interpret, and even 
question data—is an increasingly valuable skill,” says Harvard Business School Professor 
Janice Hammond in the HBS Online course Business Analytics.
With the right skills, data can allow you to gain and act on c

In [11]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print("Minimum chunk size:", min(chunk_lengths))
print("Maximum chunk size:", max(chunk_lengths))
print("Average chunk size:", round(np.mean(chunk_lengths), 2))

Minimum chunk size: 39
Maximum chunk size: 795
Average chunk size: 615.45


## 5. Generate Embeddings

An embedding converts text into a numerical vector representing its
semantic meaning.

We use:

`sentence-transformers/all-MiniLM-L6-v2`

This model produces **384-dimensional embeddings**.

The same embedding model is used for document chunks and user questions,
so they can be compared in the same vector space.

In [12]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_embedding = embedding_model.embed_query(
    "What is data analytics?"
)

print("Embedding type:", type(test_embedding))
print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Loading weights: 100%|█████████████| 103/103 [00:00<00:00, 3818.53it/s]


Embedding type: <class 'list'>
Embedding dimensions: 384
First 10 values: [-0.021909672766923904, 0.054780229926109314, -0.14216221868991852, 0.0639025941491127, -0.016469096764922142, -0.06386063247919083, 0.07600279152393341, -0.01985573209822178, -0.033540137112140656, 0.029196957126259804]


## 6. Vector Database with FAISS

The embeddings are stored in a FAISS vector database.

FAISS allows us to search for chunks whose embeddings are semantically
similar to the embedding of a user's question.

In [13]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Number of vectors in FAISS:", vector_store.index.ntotal)

Number of vectors in FAISS: 51


## 7. Semantic Search / Retrieval

When a user asks a question:

1. The question is converted into an embedding.
2. FAISS compares it with the stored chunk embeddings.
3. The most relevant chunks are returned.

We use `k=3`, meaning the top three chunks are retrieved.

In [14]:
def retrieve_chunks(question, k=3):
    results = vector_store.similarity_search_with_score(
        question,
        k=k
    )
    return results

In [15]:
question = "What is data analytics?"

results = retrieve_chunks(question, k=3)

for i, (result, score) in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Page:", result.metadata["page"])
    print("Distance score:", round(float(score), 4))
    print(result.page_content[:700])


--- Result 1 ---
Page: 3
Distance score: 0.627
Data Science 
vs. Data Analytics: 
What’s the 
Difference?
If you’re new to the world of data, two terms you’re likely to encounter 
are “data science” and “data analytics. ” While these terms are 
related, they refer to different things. Here’s an overview of 
what each term means and how it applies to business.

--- Result 2 ---
Page: 6
Distance score: 0.6729
6
Data Science vs. Data Analytics: What’s the Difference?
Data Analytics in Business
The main goal of business analytics is to extract meaningful insights from 
data that an organization can use to inform its strategy and, ultimately, 
reach its objectives. Business analytics can be used for:
• Budgeting and forecasting: By assessing a company’s historical 
revenue, sales, and costs data alongside its goals for future growth, 
an analyst can identify the budget and investments required to make 
those goals a reality.
• Risk management: By understanding the likelihood of certain 
bu

### Understanding FAISS Scores

For this FAISS setup, the returned value is a distance score.

```text
Lower distance → more similar
Higher distance → less similar
```

The retrieved text should still be checked for relevance because semantic
similarity does not always mean that a chunk contains the exact answer.

## 8. Build Retrieved Context

The retrieved chunks are combined into one context block.

Page metadata is included so that the source can be displayed with the answer.

In [16]:
def build_context(results):
    context_parts = []

    for result, score in results:
        page = result.metadata["page"]
        context_parts.append(
            f"[Page {page}]\n{result.page_content}"
        )

    return "\n\n".join(context_parts)

In [17]:
results = retrieve_chunks(
    "What is data analytics?",
    k=3
)

context = build_context(results)

print(context)

[Page 3]
Data Science 
vs. Data Analytics: 
What’s the 
Difference?
If you’re new to the world of data, two terms you’re likely to encounter 
are “data science” and “data analytics. ” While these terms are 
related, they refer to different things. Here’s an overview of 
what each term means and how it applies to business.

[Page 6]
6
Data Science vs. Data Analytics: What’s the Difference?
Data Analytics in Business
The main goal of business analytics is to extract meaningful insights from 
data that an organization can use to inform its strategy and, ultimately, 
reach its objectives. Business analytics can be used for:
• Budgeting and forecasting: By assessing a company’s historical 
revenue, sales, and costs data alongside its goals for future growth, 
an analyst can identify the budget and investments required to make 
those goals a reality.
• Risk management: By understanding the likelihood of certain 
business risks occurring—and their associated expenses—an analyst 
can make cost

## 9. Prompt Engineering

The LLM should answer from the retrieved PDF context rather than relying
on unrelated outside knowledge.

The prompt instructs the model to:

- use only the supplied context
- answer the user's question
- avoid outside knowledge
- clearly state when the answer is not found in the retrieved context

In [18]:
def create_prompt(question, context):
    prompt = f"""
You are a helpful Data Analytics assistant.

Instructions:
- Answer the question using only the provided context.
- Do not use outside knowledge.
- If the answer cannot be found in the context, say:
  "I could not find the answer in the provided document."
- Keep the answer clear and concise.
- Mention the relevant page number when possible.

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

In [19]:
question = "What is data analytics?"

results = retrieve_chunks(question, k=3)
context = build_context(results)
prompt = create_prompt(question, context)

print(prompt)


You are a helpful Data Analytics assistant.

Instructions:
- Answer the question using only the provided context.
- Do not use outside knowledge.
- If the answer cannot be found in the context, say:
  "I could not find the answer in the provided document."
- Keep the answer clear and concise.
- Mention the relevant page number when possible.

Context:
[Page 3]
Data Science 
vs. Data Analytics: 
What’s the 
Difference?
If you’re new to the world of data, two terms you’re likely to encounter 
are “data science” and “data analytics. ” While these terms are 
related, they refer to different things. Here’s an overview of 
what each term means and how it applies to business.

[Page 6]
6
Data Science vs. Data Analytics: What’s the Difference?
Data Analytics in Business
The main goal of business analytics is to extract meaningful insights from 
data that an organization can use to inform its strategy and, ultimately, 
reach its objectives. Business analytics can be used for:
• Budgeting and f

## 10. Connect the LLM

The OpenAI API key is loaded from the `.env` file.

### Security

The API key should **never** be written directly inside the notebook
or committed to GitHub.

The `.env` file must be included in `.gitignore`.

In [22]:
load_dotenv()

api_key_available = bool(os.getenv("GOOGLE_API_KEY"))

print("API key available:", api_key_available)

if not api_key_available:
    raise ValueError(
        "GOOGLE_API_KEY was not found. "
        "Add it to the .env file and restart the notebook kernel."
    )

API key available: True


In [24]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)


In [39]:
response = llm.invoke(
    "Explain data analytics in one simple sentence."
)

if isinstance(response.content, list):
    answer = "".join(
        item.get("text", "")
        for item in response.content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    answer = response.content

print(answer)

Data analytics is the process of examining raw information to find meaningful patterns and make smarter decisions.


## 11. Complete RAG Pipeline

The complete RAG pipeline combines retrieval, context construction,
prompt creation, and LLM generation.

```text
User Question
      ↓
Semantic Retrieval
      ↓
Relevant PDF Chunks
      ↓
Context
      ↓
Prompt
      ↓
LLM
      ↓
Answer + Source Pages
```

In [37]:
def ask_rag(question, k=3):
    # retrieve relevant chunks
    results = retrieve_chunks(question, k=k)

    # build context
    context = build_context(results)

    # create prompt
    prompt = create_prompt(question, context)

    # generate answer
    response = llm.invoke(prompt)

    # extract only the answer text
    if isinstance(response.content, list):
        answer = "".join(
            item.get("text", "")
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        answer = response.content

    # collect source pages
    source_pages = sorted(
        set(result.metadata["page"] for result, score in results)
    )

    # collect retrieval distances
    retrieval_scores = [
        round(float(score), 4)
        for result, score in results
    ]

    return {
        "answer": answer,
        "source_pages": source_pages,
        "retrieval_scores": retrieval_scores,
        "retrieved_chunks": results
    }

In [38]:
question = "What is data analytics?"

result = ask_rag(question)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

print("\nSource pages:")
print(result["source_pages"])

print("\nRetrieval distances:")
print(result["retrieval_scores"])

Question:
What is data analytics?

Answer:
Data analytics refers to the process and practice of analyzing data to answer questions, extract insights, and identify trends (Page 4).

Source pages:
[3, 4, 6]

Retrieval distances:
[0.627, 0.6729, 0.6848]


## 12. Interactive Chatbot

The following function provides a simple terminal-based interactive chatbot.

Type `exit` or `quit` to stop the conversation.

Each question goes through the same RAG pipeline used above.

In [40]:
def chatbot():
    print("=" * 70)
    print("                 MINI RAG CHATBOT")
    print("=" * 70)
    print("Ask questions about the Data & Analytics PDF.")
    print("Type 'exit' or 'quit' to end the conversation.")
    print("=" * 70)

    while True:
        question = input("\nYou: ").strip()

        if question.lower() in {"exit", "quit"}:
            print("\nChatbot: Goodbye!")
            break

        if not question:
            print("Chatbot: Please enter a question.")
            continue

        result = ask_rag(question)

        print("\nChatbot:")
        print(result["answer"])
        print("\nSource pages:", result["source_pages"])

### Run the Interactive Chatbot

The next cell is intentionally commented out so that the notebook does not
wait for input when it is run from top to bottom.

Uncomment `chatbot()` when you want to interact with the PDF.

In [41]:
chatbot()

                 MINI RAG CHATBOT
Ask questions about the Data & Analytics PDF.
Type 'exit' or 'quit' to end the conversation.



You:  what is data literacy?



Chatbot:
Data literacy is the ability to read, understand, and utilize data in different ways (Page 7).

Source pages: [2, 7, 21]



You:  how we can use data science?



Chatbot:
Based on the provided document, in business, data science is used to collect, organize, and maintain data—often to write algorithms for large-scale analysis. Specifically, you can use data science to:

* **Gain customer insights:** Reveal details about customer habits, demographics, preferences, and aspirations to improve user experiences and inform retargeting efforts (Page 5).
* **Increase security:** Protect sensitive information, such as using machine-learning algorithms to detect bank fraud faster and more accurately than humans (Page 5).
* **Inform internal finances:** Help financial teams create reports, generate forecasts, and analyze financial trends (Page 5).
* **Set up for success:** Combine business know-how and intuition with data science to minimize risks, increase profits, and set your company up for success (Page 5, Page 15).

Source pages: [5, 15]


...


Chatbot:
Based on the provided document, to determine which data and analytics course is right for you, you should take stock of your current data and analytics knowledge and your future goals, then explore the course comparison table (Page 19).

Source pages: [1, 19, 20]



You:  what is machine learning?



Chatbot:
Based on the provided document, machine learning refers to the use of computer algorithms that automatically learn from and adapt in response to data (Page 14) to find structure in data and make predictions (Page 12).

Source pages: [12, 14]



You:  exit



Chatbot: Goodbye!


## 13. Evaluation

A RAG system should be tested with different types of questions.

We evaluate:

1. Questions clearly answered in the PDF
2. Paraphrased questions
3. Questions requiring information from more than one retrieved chunk
4. Questions whose answers are not in the document
5. Retrieval quality and the effect of different `k` values

In [43]:
# Test 1: questions clearly answered in the PDF

test_questions = [
    "What is data analytics?",
    "What is data science?"
    
]

for question in test_questions:
    result = ask_rag(question)

    print("\n" + "=" * 70)
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Source pages:", result["source_pages"])

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 20.411759857s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [ ]:
# Test 2: paraphrased / semantic questions

semantic_questions = [
    "How can analyzing data help an organization make decisions?",
    "Why is understanding data important for business professionals?"
]

for question in semantic_questions:
    result = ask_rag(question)

    print("\n" + "=" * 70)
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Source pages:", result["source_pages"])

In [ ]:
# Test 3: question that should not be answered from the PDF

question = "What is the capital of France?"

result = ask_rag(question)

print("Question:", question)
print("\nAnswer:")
print(result["answer"])

print("\nSource pages:")
print(result["source_pages"])

### Expected behavior for the out-of-document test

For a question whose answer is not supported by the PDF, the chatbot
should respond with:

```text
I couldn't find the answer in the provided document.
```

The exact wording may vary slightly depending on the LLM response.
The important requirement is that the model should not invent information
from outside the retrieved context.

In [ ]:
# Test 4: inspect retrieval quality

question = "What is data analytics?"

results = retrieve_chunks(question, k=3)

for i, (result, score) in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Page:", result.metadata["page"])
    print("Distance:", round(float(score), 4))
    print("Text:", result.page_content[:500])

In [ ]:
# Test 5: compare different values of k

question = "What is data analytics?"

for k in [2, 3, 5]:
    results = retrieve_chunks(question, k=k)

    print("\n" + "=" * 70)
    print("k =", k)

    for i, (result, score) in enumerate(results, start=1):
        print(
            f"Result {i}: "
            f"Page {result.metadata['page']} | "
            f"Distance {float(score):.4f}"
        )

## Evaluation Summary

The Mini RAG Chatbot was evaluated using different types of questions
to check retrieval quality, answer generation, source tracking, and
the ability to handle questions that are not covered by the document.

| Test | Question | Result | Source Pages |
|---|---|---|---|
| Direct question | What is data literacy? | Correctly answered from the document | 2, 7, 21 |
| Application question | How we can use data science? | Provided a detailed answer based on retrieved content | 5, 15 |
| Out-of-document question | What are the 4 types of analytics? | Incorrectly indicated that the answer was not found | 5, 6 |
| Concept question | What is data ecosystem? | Correctly explained using retrieved document content | 7, 8, 21 |
| Skill-development question | How to improve your skills? | Retrieved relevant recommendations from the document | 11, 16, 17 |
| Course recommendation | Which data and analytics course is right for you? | Correctly retrieved the relevant course-selection section | 1, 19, 20 |
| Technical concept | What is machine learning? | Correctly explained using information from the document | 12, 14 |

### Evaluation Findings

**1. Retrieval Performance**

The chatbot successfully retrieved relevant chunks for the tested questions.
For example, questions about data literacy, data science, data ecosystem,
and machine learning returned relevant source pages from the PDF.

**2. Answer Generation**

The LLM generated meaningful answers using the retrieved context.
For questions such as "How we can use data science?" and "How to improve
your skills?", the chatbot combined information from multiple retrieved
pages to produce a detailed response.

**3. Source Traceability**

The chatbot displays the source pages used to generate each response.
This makes the generated answers easier to verify against the original PDF.

**4. Out-of-Document Handling**

The question:

"What are the 4 types of analytics?"

was not answered using outside knowledge. Instead, the chatbot responded:

"I could not find the answer in the provided document."

This demonstrates the grounding instruction used in the RAG prompt and
helps reduce unsupported answers.

**5. Overall Result**

The RAG pipeline successfully performed the complete workflow:

PDF → Chunking → Embeddings → FAISS → Semantic Retrieval → Context
→ Prompt → LLM → Answer

The chatbot was able to answer different types of questions from the
document while providing source pages and handling an unavailable answer
appropriately.

## 15. Conclusion

This project demonstrates a complete end-to-end Retrieval-Augmented
Generation pipeline for question answering over a PDF document.

The system:

- extracts text from the PDF
- splits the text into chunks
- generates semantic embeddings
- stores embeddings in FAISS
- retrieves relevant chunks for a question
- constructs a context-aware prompt
- sends the context to an LLM
- generates a final answer
- displays source pages
- provides an interactive chatbot
- evaluates retrieval and question-answering behavior

### Final Pipeline

```text
PDF
 ↓
Text Extraction
 ↓
22 Pages
 ↓
51 Chunks
 ↓
384-Dimensional Embeddings
 ↓
FAISS Vector Database
 ↓
Semantic Retrieval
 ↓
Context
 ↓
Prompt Engineering
 ↓
OpenAI LLM
 ↓
Answer + Source Pages
```